<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [2]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [3]:
import folium
import pandas as pd

In [4]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [ ]:
## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [5]:
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

Now, you can take a look at what are the coordinates for each site.


In [6]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [7]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [8]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [9]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label
for index, site in launch_sites_df.iterrows():
    # Get the coordinates and site name
    site_coordinate = [site['Lat'], site['Long']]
    site_name = site['Launch Site']
    
    # Add a circle for the launch site
    folium.Circle(
        site_coordinate,
        radius=1000,
        color='#000000',
        fill=True
    ).add_child(folium.Popup(site_name)).add_to(site_map)
    
    # Add a marker with the launch site name as a text label
    folium.map.Marker(
        site_coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 12; color:#d35400;"><b>{site_name}</b></div>'
        )
    ).add_to(site_map)

# Display the map
site_map

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


In [ ]:
# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [10]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [11]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [12]:
# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red

# Create a function to determine marker color
def get_marker_color(launch_class):
    if launch_class == 1:
        return 'green'
    else:
        return 'red'

# Apply the function to create the new marker_color column
spacex_df['marker_color'] = spacex_df['class'].apply(get_marker_color)

# Display first few rows to verify the new column
print(spacex_df.head())

   Launch Site        Lat       Long  class marker_color
0  CCAFS LC-40  28.562302 -80.577356      0          red
1  CCAFS LC-40  28.562302 -80.577356      0          red
2  CCAFS LC-40  28.562302 -80.577356      0          red
3  CCAFS LC-40  28.562302 -80.577356      0          red
4  CCAFS LC-40  28.562302 -80.577356      0          red


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [13]:
# First, make sure we have a marker_cluster object
marker_cluster = MarkerCluster().add_to(site_map)

# Iterate through each row in spacex_df
for index, record in spacex_df.iterrows():
    # Create a Marker object with its coordinate
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        # Customize the Marker's icon property using the marker_color column
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        # Add a popup with launch information
        popup=folium.Popup(
            f"Launch Site: {record['Launch Site']}<br>"
            f"Outcome: {'Success' if record['class'] == 1 else 'Failure'}<br>"
            f"Latitude: {record['Lat']}<br>"
            f"Longitude: {record['Long']}",
            max_width=300
        )
    )
    
    # Add the marker to the marker cluster
    marker_cluster.add_child(marker)

# Display the map
site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [ ]:
# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [14]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [15]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [16]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
# For CCAFS LC-40 / SLC-40 (Cape Canaveral)
# Create an empty dictionary to store distances
site_distances = {}

# Coastline coordinates for each launch site
coastline_points = {
    'CCAFS LC-40': {'lat': 28.56367, 'lon': -80.57163, 'name': 'Cape Canaveral Coast'},
    'CCAFS SLC-40': {'lat': 28.56367, 'lon': -80.57163, 'name': 'Cape Canaveral Coast'},
    'KSC LC-39A': {'lat': 28.60820, 'lon': -80.60370, 'name': 'Kennedy Space Center Coast'},
    'VAFB SLC-4E': {'lat': 34.63250, 'lon': -120.61100, 'name': 'Vandenberg Coast'}
}

# Calculate distances for each launch site
for index, site in launch_sites_df.iterrows():
    site_name = site['Launch Site']
    launch_site_lat = site['Lat']
    launch_site_lon = site['Long']
    
    # Get coastline coordinates for this site
    if site_name in coastline_points:
        coastline_lat = coastline_points[site_name]['lat']
        coastline_lon = coastline_points[site_name]['lon']
        coastline_name = coastline_points[site_name]['name']
        
        # Calculate distance using the provided function
        distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
        
        # Store the distance in the dictionary
        site_distances[site_name] = {
            'distance': distance_coastline,
            'launch_coord': [launch_site_lat, launch_site_lon],
            'coast_coord': [coastline_lat, coastline_lon],
            'coast_name': coastline_name
        }
        
        print(f"{site_name}: Distance to {coastline_name} = {distance_coastline:.2f} km")
        print(f"  Launch Site: ({launch_site_lat:.5f}, {launch_site_lon:.5f})")
        print(f"  Coastline: ({coastline_lat:.5f}, {coastline_lon:.5f})")
        print()

CCAFS LC-40: Distance to Cape Canaveral Coast = 0.58 km
  Launch Site: (28.56230, -80.57736)
  Coastline: (28.56367, -80.57163)

CCAFS SLC-40: Distance to Cape Canaveral Coast = 0.51 km
  Launch Site: (28.56320, -80.57682)
  Coastline: (28.56367, -80.57163)

KSC LC-39A: Distance to Kennedy Space Center Coast = 5.74 km
  Launch Site: (28.57325, -80.64690)
  Coastline: (28.60820, -80.60370)

VAFB SLC-4E: Distance to Vandenberg Coast = 0.04 km
  Launch Site: (34.63283, -120.61075)
  Coastline: (34.63250, -120.61100)



In [17]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
# for example
# distance_marker = folium.Marker(
#    coordinate,
#    icon=DivIcon(
#        icon_size=(20,20),
#        icon_anchor=(0,0),
#        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
#        )
#    )

# Create and add markers for each launch site's coastline distance
for site_name, site_data in site_distances.items():
    launch_coord = site_data['launch_coord']
    coast_coord = site_data['coast_coord']
    distance = site_data['distance']
    coast_name = site_data['coast_name']
    
    # 1. Add a marker at the coastline point
    folium.Marker(
        location=coast_coord,
        icon=folium.Icon(color='blue', icon='information-sign'),
        popup=folium.Popup(
            f"<b>{coast_name}</b><br>"
            f"Coordinates: {coast_coord[0]:.5f}, {coast_coord[1]:.5f}<br>"
            f"<b>Distance to {site_name}: {distance:.2f} km</b>",
            max_width=300
        )
    ).add_to(site_map)
    
    # 2. Add a marker at the midpoint showing the distance
    midpoint = [(launch_coord[0] + coast_coord[0])/2, (launch_coord[1] + coast_coord[1])/2]
    
    # Using the exact format from the example
    distance_marker = folium.Marker(
        midpoint,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
        )
    )
    site_map.add_child(distance_marker)
    
    # 3. Draw a line connecting launch site to coastline
    folium.PolyLine(
        locations=[launch_coord, coast_coord],
        color='#d35400',
        weight=2,
        opacity=0.7,
        popup=f"Distance: {distance:.2f} km"
    ).add_to(site_map)

# Display the updated map
site_map

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [ ]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
# lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [18]:
# For CCAFS / KSC area (Florida)
railway_ccafs = [28.572, -80.655]  # Approximate railway line near CCAFS
highway_ccafs = [28.562, -80.738]  # US-1 highway
city_titusville = [28.612, -80.808]  # Titusville, FL (nearest city)

# For VAFB area (California)
railway_vafb = [34.643, -120.578]  # Railway near VAFB
highway_vafb = [34.675, -120.553]  # Highway 1 near VAFB
city_lompoc = [34.639, -120.458]  # Lompoc, CA

# Alternatively, you can use these more precise coordinates if you have them:
# After using MousePosition, replace with your actual found coordinates

# Dictionary to store all points of interest
poi_coordinates = {
    'CCAFS LC-40': {
        'railway': {'lat': 28.572, 'lon': -80.655, 'name': 'Railway near CCAFS'},
        'highway': {'lat': 28.562, 'lon': -80.738, 'name': 'US-1 Highway'},
        'city': {'lat': 28.612, 'lon': -80.808, 'name': 'Titusville, FL'}
    },
    'CCAFS SLC-40': {
        'railway': {'lat': 28.572, 'lon': -80.655, 'name': 'Railway near CCAFS'},
        'highway': {'lat': 28.562, 'lon': -80.738, 'name': 'US-1 Highway'},
        'city': {'lat': 28.612, 'lon': -80.808, 'name': 'Titusville, FL'}
    },
    'KSC LC-39A': {
        'railway': {'lat': 28.572, 'lon': -80.655, 'name': 'Railway near KSC'},
        'highway': {'lat': 28.562, 'lon': -80.738, 'name': 'US-1 Highway'},
        'city': {'lat': 28.612, 'lon': -80.808, 'name': 'Titusville, FL'}
    },
    'VAFB SLC-4E': {
        'railway': {'lat': 34.643, 'lon': -120.578, 'name': 'Railway near VAFB'},
        'highway': {'lat': 34.675, 'lon': -120.553, 'name': 'Highway 1'},
        'city': {'lat': 34.639, 'lon': -120.458, 'name': 'Lompoc, CA'}
    }
}

# Color scheme for different types of lines
line_colors = {
    'railway': '#FF4444',  # Red
    'highway': '#44FF44',  # Green
    'city': '#4444FF'      # Blue
}

# Marker icons for different POIs
poi_icons = {
    'railway': folium.Icon(color='red', icon='train', prefix='fa'),
    'highway': folium.Icon(color='green', icon='road', prefix='fa'),
    'city': folium.Icon(color='blue', icon='building', prefix='fa')
}

# Calculate and add lines for each launch site
for index, site in launch_sites_df.iterrows():
    site_name = site['Launch Site']
    launch_lat = site['Lat']
    launch_lon = site['Long']
    
    if site_name in poi_coordinates:
        site_pois = poi_coordinates[site_name]
        
        # For each type of POI (railway, highway, city)
        for poi_type, poi_info in site_pois.items():
            poi_lat = poi_info['lat']
            poi_lon = poi_info['lon']
            poi_name = poi_info['name']
            
            # Calculate distance
            distance = calculate_distance(launch_lat, launch_lon, poi_lat, poi_lon)
            
            # Add marker for the POI
            folium.Marker(
                location=[poi_lat, poi_lon],
                icon=poi_icons[poi_type],
                popup=folium.Popup(
                    f"<b>{poi_name}</b><br>"
                    f"Type: {poi_type.title()}<br>"
                    f"Coordinates: {poi_lat:.5f}, {poi_lon:.5f}<br>"
                    f"<b>Distance to {site_name}: {distance:.2f} km</b>",
                    max_width=300
                )
            ).add_to(site_map)
            
            # Add distance marker at midpoint
            midpoint = [(launch_lat + poi_lat)/2, (launch_lon + poi_lon)/2]
            
            distance_marker = folium.Marker(
                midpoint,
                icon=DivIcon(
                    icon_size=(20,20),
                    icon_anchor=(0,0),
                    html=f'<div style="font-size: 10; color:{line_colors[poi_type]};"><b>{distance:.2f} KM</b></div>'
                )
            )
            site_map.add_child(distance_marker)
            
            # Draw line from launch site to POI
            folium.PolyLine(
                locations=[[launch_lat, launch_lon], [poi_lat, poi_lon]],
                color=line_colors[poi_type],
                weight=2,
                opacity=0.7,
                popup=f"{poi_type.title()} Distance: {distance:.2f} km"
            ).add_to(site_map)
            
            print(f"{site_name} → {poi_name}: {distance:.2f} km")

# Alternative approach: If you want to find coordinates interactively first
# You can run this cell to get coordinates by clicking on the map

from folium import LatLngPopup

# Add click functionality to get coordinates
site_map.add_child(LatLngPopup())

# Display the map with all new markers and lines
site_map


CCAFS LC-40 → Railway near CCAFS: 7.66 km
CCAFS LC-40 → US-1 Highway: 15.69 km
CCAFS LC-40 → Titusville, FL: 23.20 km
CCAFS SLC-40 → Railway near CCAFS: 7.70 km
CCAFS SLC-40 → US-1 Highway: 15.75 km
CCAFS SLC-40 → Titusville, FL: 23.22 km
KSC LC-39A → Railway near KSC: 0.80 km
KSC LC-39A → US-1 Highway: 8.99 km
KSC LC-39A → Titusville, FL: 16.31 km
VAFB SLC-4E → Railway near VAFB: 3.20 km
VAFB SLC-4E → Highway 1: 7.06 km
VAFB SLC-4E → Lompoc, CA: 14.00 km


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
